# MrBilit Search Data Analysis

This notebook explores real-world search logs from the MrBilit travel platform.

The analysis covers:
- service popularity,
- normalization of terminal-specific destination names,
- most-searched cities,
- most-searched provinces,
- highly populated cities that are absent from the top search list.


In [ ]:
import pandas as pd
import plotly.express as px


## 1. Load the data

`mrbilit_search.json` contains user search activity and `iran_cities.csv`
contains city metadata used to enrich the analysis.


In [ ]:
data = pd.read_json("mrbilit_search.json")
cities = pd.read_csv("iran_cities.csv")

print("Search data shape:", data.shape)
print("Cities data shape:", cities.shape)

data.head()


## 2. Service popularity

The share of each `ServiceType` is calculated as a fraction between 0 and 1.


In [ ]:
percentages = data["ServiceType"].value_counts(normalize=True).to_dict()
percentages


In [ ]:
fig = px.pie(
    names=percentages.keys(),
    values=percentages.values(),
    title="Service Type Popularity"
)
fig.show()


## 3. Destination preprocessing

Some accepted destinations include terminal details such as
`شیراز - پایانه کاراندیش`. For city-level analysis, only the part before
the first hyphen is retained and surrounding whitespace is removed.


In [ ]:
preprocessed_data = data.copy()

mask = preprocessed_data["AcceptString"].str.contains("-", na=False)

preprocessed_data.loc[mask, "AcceptString"] = (
    preprocessed_data.loc[mask, "AcceptString"]
    .str.split("-", n=1)
    .str[0]
    .str.strip()
)

preprocessed_data.head()


## 4. Most-searched cities

Hotel searches are excluded because their accepted strings may refer to
properties rather than transportation destinations.


In [ ]:
data_transport = preprocessed_data[
    preprocessed_data["ServiceType"] != "hotel"
].copy()

top_cities = (
    data_transport["AcceptString"]
    .value_counts()
    .head(20)
    .index
    .tolist()
)

top_cities


In [ ]:
top_city_rows = data_transport[
    data_transport["AcceptString"].isin(top_cities)
]

fig = px.histogram(
    top_city_rows,
    x="AcceptString",
    title="Top 20 Most-Searched Cities"
)
fig.show()


## 5. Most-searched provinces

Search rows are joined to the city dataset using the Persian city name.
An inner join is used so only destinations present in the city reference
dataset are included.


In [ ]:
merged = pd.merge(
    data_transport,
    cities,
    left_on="AcceptString",
    right_on="City FA",
    how="inner"
)

top_provinces = (
    merged["Province"]
    .value_counts()
    .head(15)
    .index
    .tolist()
)

top_provinces


## 6. Population vs. search popularity

Find cities with a 2016 census population above 500,000 that do not
appear in the top 20 searched cities.


In [ ]:
not_in_top = cities[
    (cities["2016 Census"] > 500_000) &
    (~cities["City FA"].isin(top_cities))
]["City FA"].tolist()

not_in_top


## Key findings from the completed run

- The largest service share was **bus (~40.24%)**, followed by **flight (~24.96%)**.
- The top searched cities began with **Tehran, Mashhad, Isfahan, Shiraz, and Ahvaz**.
- The top searched provinces began with **Tehran, Razavi Khorasan, Khuzestan, Fars, and Hormozgan**.
- Highly populated cities outside the top-20 search list were **Ardabil, Hamedan, and Urmia**.

The notebook is intentionally focused on reproducible analysis rather than the
competition-specific result-packaging cells from the original exercise.
